In [23]:
import pandas as pd
from astroquery.gaia import Gaia
from astropy.table import Table
import numpy as np

In [2]:
def clean_jobids():

    # Access to the Gaia Archive as registered user
    #Gaia.login()
    # Retrieve job's metadata - this may take a few minutes, depending on the number of jobs stored
    job_ids = [job.jobid for job in Gaia.list_async_jobs()]
    print('job ids',job_ids)
    # Gaia.list_async_jobs() method retrieves a "job" object that contains the job ID of each asynchronous job stored in the user account.
    # Uncomment the following line if you want to delete all the jobs stored in the user space and stop the code there. See below how to delete a subset of the jobs.
    Gaia.remove_jobs(job_ids)

In [10]:
def get_gaia_sample(catalog="gaiadr3.gold_sample_oba_stars"):
    """
    To grab the catalog star sample
    """

    query = "SELECT * FROM {}".format(catalog)

    job     = Gaia.launch_job_async(query,verbose=True)
    #job     = Gaia.launch_job(query)
    results = job.get_results()
    print(f'Table size (rows): {len(results)}')
    #results
    return results.to_pandas()

    

In [4]:
def clean_gaia_sample(targets):
    """
    Select OBA stars according to the pub A&A 674,A39 (2023) (Gaia Data Release 3: A golden sample of astrophysical parameters)
    """

    print(targets)

    # select as in paper
    idx = targets['vtan_flag'] == 0

    sel = targets[idx]

    print(len(targets),len(sel))

    return sel

In [5]:
def get_gaia_from_db(ra_min,ra_max,gaiadr='gaiadr3',catname='gaia_source'):
    # Main query, get selected infos with criteria.
    # All data information are given in the Gaia data model: 
    # https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html

    vars = 'source_id, ra, dec, pmra, pmdec, parallax, parallax_error, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, l, b, phot_variable_flag, ref_epoch'
    vars += ', L, B'
    
    if gaiadr == 'gaiadr2':
        vars += ', a_g_val'
    if gaiadr == 'gaiadr3':
        vars += ', ag_gspphot'

    """
    query = "SELECT {} \
    FROM {}.gaia_source \
    WHERE visibility_periods_used > 5 \
    AND astrometric_excess_noise < 0.5 \
    AND parallax > 1 \
    AND parallax_over_error > 5 \
    AND phot_bp_mean_flux_over_error > 20 \
    AND phot_rp_mean_flux_over_error > 20 \
    AND phot_g_mean_flux_over_error > 50 \
    AND phot_bp_rp_excess_factor < 1.2*(1.2+0.03*power(phot_bp_mean_mag-phot_rp_mean_mag,2)) \
    AND ra >= {} \
    and ra < {}".format(vars,gaiadr,ra_min,ra_max)
    """
    query = "SELECT {} \
    FROM {}.{} \
    WHERE visibility_periods_used > 8 \
    AND parallax_over_error > 10 \
    AND phot_bp_mean_flux_over_error > 20 \
    AND phot_rp_mean_flux_over_error > 20 \
    AND phot_g_mean_flux_over_error > 50 \
    AND phot_bp_rp_excess_factor < 1.3+0.06*power(phot_bp_mean_mag-phot_rp_mean_mag,2) \
    AND phot_bp_rp_excess_factor > 1.0+0.015*power(phot_bp_mean_mag-phot_rp_mean_mag,2) \
    AND astrometric_chi2_al/(astrometric_n_good_obs_al-5) < 1.44*greatest(1,exp(-0.4*(phot_g_mean_mag-19.5))) \
    AND ra >= {} \
    and ra < {}".format(vars,gaiadr,catname,ra_min,ra_max)    
    
    print(query)
    job     = Gaia.launch_job_async(query,verbose=True)
    #job     = Gaia.launch_job(query)
    results = job.get_results()
    print(f'Table size (rows): {len(results)}')
    results
    return results.to_pandas()

In [6]:
def get_sourceids(dr='gaiadr3',catalog='astrophysical_parameters'):

    vars = 'source_id'

    query = "SELECT {} FROM {}.{}".format(vars,dr,catalog)

    job     = Gaia.launch_job_async(query,verbose=True)
    #job     = Gaia.launch_job(query)
    results = job.get_results()
    print(f'Table size (rows): {len(results)}')
    results
    return results.to_pandas()

In [7]:
def get_gaia_from_dr(ra_min,ra_max,gaiadr='gaiadr3',catname='gaia_source'):
    """ 
    To get stars from a Gaia DR
    """
    
    tt = get_gaia_from_db(ra_min, ra_max,gaiadr,catname)
    ag = 'a_g_val'
    if gaiadr == 'gaiadr3':
        ag = 'ag_gspphot'
    # absolute mag G band - see Gaia Data Release 2 Documentation
    tt['MG'] = tt['phot_g_mean_mag']+5-5*np.log10(1.e3/tt['parallax'])-tt[ag]
    return tt

In [8]:
def get_targets(drs='gaiadr3',catname='gaia_source'):
    """
    To grab stars from the Gaia cat. The sky is splitted in RA slices to avoid memory pbs
    """
    
    ramin = 0
    ramax = 360
    delta_ra = 6
    ras = np.arange(ramin,ramax,delta_ra)
    outDir = '/home/philippe/LSST/gaia_files/{}/{}'.format(drs,catname)
    from sn_tools.sn_io import checkDir
    checkDir(outDir)
    for ra in ras:
        ra_min = np.round(ra,1)
        ra_max = np.round(ra_min+delta_ra,1)
        print(ra_min,ra_max)
        df = get_gaia_from_dr(ra_min,ra_max,drs,catname)
        out_name = '{}/sources_{}_{}.hdf5'.format(outDir,ra_min,ra_max)
        df.to_hdf(out_name,key='star')
        
        

In [9]:
def save_df(rr,drs,catname,key='oba_stars'):
    """
    To save a catalog
    """
    outDir = '/home/philippe/LSST/gaia_files/{}/{}'.format(drs,catname)
    from sn_tools.sn_io import checkDir
    checkDir(outDir)
    outName = 'sources_{}.hdf5'.format(catname)
    rr.to_hdf('{}/{}'.format(outDir,outName),key=key)
    

In [12]:
def get_oba_sources(catalog="gaiadr3.gold_sample_oba_stars"):
    tt_spl = table.split('.')
    drs = tt_spl[0]
    catname = tt_spl[1]
    rr = get_gaia_sample(catalog=catalog)
    save_df(rr,drs,catname,'oba_stars')
    

In [13]:
def get_fgkm_sources(catalog="gaiadr3.gold_sample_fgkm_stars"):
    tt_spl = table.split('.')
    drs = tt_spl[0]
    catname = tt_spl[1]
    rr = get_gaia_sample(catalog=catalog)
    tt = rr['evolstage_flame_spec'].unique()
    # some treatment to avoid Int32 pb...
    rr_cp = pd.DataFrame(rr)
    rr_cp["evolstage_flame_spec"] = rr_cp["evolstage_flame_spec"].apply(lambda x:-1 if x is pd.NA else x)
    #print(rr_cp['evolstage_flame_spec'].unique())
    rr_cp = rr_cp.fillna(int(-999))
    rr_cp["spectraltype_esphs"] = rr_cp["spectraltype_esphs"].apply(lambda x:'U' if x=='' else x)
    print(rr_cp['spectraltype_esphs'].unique())
    rr_cp['spectraltype_esphs'] = rr_cp['spectraltype_esphs'].astype(str)
    rr_cp['evolstage_flame'] = rr_cp['evolstage_flame'].astype(int)
    print(rr_cp.dtypes)
    save_df(rr_cp,drs,catname,'fgkm_stars')

In [31]:
def get_meta(df,Gaia,cols='ap.*'):
    
    table_id = Table([list(df['source_id'])], names=['gaia_id'], meta={'meta':'table'})
    Gaia.upload_table(upload_resource=table_id, table_name='tableid')

    query="SELECT {} \
    FROM gaiadr3.astrophysical_parameters AS ap \
    JOIN user_pgris.tableid as usert ON usert.gaia_id = ap.source_id".format(cols)

    #launch query and save and return a Dataframe
    job = Gaia.launch_job_async(query)
    results = job.get_results().to_pandas()

    # suggest delete your table from your personal space
    Gaia.delete_user_table(table_name='tableid')
    return results

In [34]:
Gaia.login(user='pgris', password='Lsst!!2024a=+')

INFO: Login to gaia TAP server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


In [34]:
#grab OBA catalog and save it on disk
# get_oba_sources()

Launched query: 'SELECT * FROM gaiadr3.gold_sample_oba_stars'
------>https
host = gea.esac.esa.int:443
context = /tap-server/tap/async
Content-type = application/x-www-form-urlencoded
303 303
[('Date', 'Tue, 21 Oct 2025 06:42:26 GMT'), ('Server', 'Apache/2.4.6 (SLES Expanded Support platform 7) OpenSSL/1.0.2k-fips mod_jk/1.2.43'), ('X-VO-Authenticated', 'pgris'), ('Location', 'https://gea.esac.esa.int/tap-server/tap/async/1761028946091O'), ('Cache-Control', 'no-cache, no-store, max-age=0, must-revalidate'), ('Pragma', 'no-cache'), ('Expires', '0'), ('X-XSS-Protection', '1; mode=block'), ('X-Frame-Options', 'SAMEORIGIN'), ('X-Content-Type-Options', 'nosniff'), ('Transfer-Encoding', 'chunked'), ('Content-Type', 'text/plain;charset=ISO-8859-1')]
job 1761028946091O, at: https://gea.esac.esa.int/tap-server/tap/async/1761028946091O
Retrieving async. results...
INFO: Query finished. [astroquery.utils.tap.core]
Table size (rows): 3023388


In [14]:
#grab FGKM stars and dump on disk
# get_fgkm_sources()

In [16]:
# get OBA source_id's
gaiaDir = '../../gaia_files'
gaiadr = 'gaiadr3'
oba_cat = 'gold_sample_oba_stars/sources_gold_sample_oba_stars.hdf5'

oba_stars = pd.read_hdf('{}/{}/{}'.format(gaiaDir,gaiadr,oba_cat))

oba_stars = clean_gaia_sample(oba_stars)
len(oba_stars)

                   source_id  vtan_flag
0         243212732278284416          0
1         243215068740428672          0
2         243222662242499840          0
3         243225651539759104          0
4         243225685899488640          0
...                      ...        ...
3023383  5959547056346917248          0
3023384  5959547567393419392          0
3023385  5959547670472310528          0
3023386  5959549255370331392          0
3023387  5959549392809229824          0

[3023388 rows x 2 columns]
3023388 2746935


2746935

In [19]:
oba_stars['source_id'].to_list()

[243212732278284416,
 243215068740428672,
 243222662242499840,
 243225651539759104,
 243225685899488640,
 243230461903072896,
 243232008091234816,
 243234481992427904,
 243236955893487616,
 243239395434866688,
 243241250861185408,
 243243827841075584,
 243245270950574336,
 243246507900622848,
 243248981801940096,
 243249806435630848,
 243257262499813632,
 243259633320654464,
 243262107221824384,
 243263447251579264,
 243268291975086336,
 243268841730865536,
 243270005662782976,
 243270181760227456,
 243272758740584448,
 243273480295064960,
 243274236209738496,
 243274889045633024,
 243275232642143744,
 243275713678442240,
 243281589193278080,
 243283994375039360,
 243289350195028608,
 243289968670451840,
 243292481230350080,
 243294267936710144,
 243302926591041792,
 243308561588092288,
 243308699027029632,
 243311344726854528,
 243313200152769536,
 243315536614919808,
 243315536614920448,
 243320411398938496,
 243322717801348096,
 243325432219353984,
 243325466579090688,
 243325569658

In [24]:
# make a request to grab available columns
rr = get_meta(oba_stars[:10],Gaia)
cols = rr.columns.to_list()
cols = list(filter(lambda s: not ('oa' in s), cols))
cols_db = ','.join(cols)
cols_db

INFO: Sending pytable. [astroquery.utils.tap.core]
INFO: Uploaded table 'tableid'. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]
INFO: Table 'tableid' deleted. [astroquery.utils.tap.core]


,solution_id,source_id,classprob_dsc_combmod_quasar,classprob_dsc_combmod_galaxy,classprob_dsc_combmod_star,classprob_dsc_combmod_whitedwarf,classprob_dsc_combmod_binarystar,classprob_dsc_specmod_quasar,classprob_dsc_specmod_galaxy,classprob_dsc_specmod_star,...,ag_msc_upper,ag_msc_lower,logposterior_msc,mcmcaccept_msc,mcmcdrift_msc,flags_msc,neuron_oa_id,neuron_oa_dist,neuron_oa_dist_percentile_rank,flags_oa
0,1636148068921376768,243212732278284416,1.022396e-13,5.106816e-13,0.999979,5.342136e-10,0.000021,0.000000,0.0,0.997854,...,0.444588,0.122586,-176.645203,0.174511,0.067618,0,<NA>,NaN,<NA>,
1,1636148068921376768,243215068740428672,5.124280e-11,5.103790e-13,0.999985,2.387608e-10,0.000015,0.000005,0.0,0.998451,...,5.000000,0.000000,-698.042114,0.131050,0.010333,0,<NA>,NaN,<NA>,
2,1636148068921376768,243222662242499840,2.685261e-11,5.104969e-13,0.999982,3.322484e-10,0.000018,0.000003,0.0,0.998218,...,1.968054,0.506835,-334.070557,0.115704,0.016557,0,<NA>,NaN,<NA>,
3,1636148068921376768,243225651539759104,1.023207e-13,5.110869e-13,0.999970,8.197416e-10,0.000030,0.000000,0.0,0.997054,...,0.956783,0.493951,-1213.568848,0.168067,0.059471,1,<NA>,NaN,<NA>,
4,1636148068921376768,243225685899488640,1.200140e-10,5.122913e-13,0.999947,9.994340e-09,0.000053,0.000012,0.0,0.994687,...,0.241910,0.000000,-977.054382,0.199928,0.046965,0,<NA>,NaN,<NA>,
5,1636148068921376768,243230461903072896,1.022697e-13,5.108319e-13,0.999976,5.619797e-10,0.000024,0.000000,0.0,0.997557,...,2.100900,0.204877,-4772.345215,0.156778,0.052380,1,<NA>,NaN,<NA>,
6,1636148068921376768,243232008091234816,6.965268e-11,5.110064e-13,0.999972,2.148561e-09,0.000028,0.000007,0.0,0.997213,...,0.558281,0.511669,-3695.009766,0.236874,0.763130,1,<NA>,NaN,<NA>,
7,1636148068921376768,243234481992427904,6.058117e-11,5.107268e-13,0.999978,1.028625e-09,0.000022,0.000006,0.0,0.997765,...,0.744663,0.000000,-5678.925293,0.174702,0.022446,1,<NA>,NaN,<NA>,
8,1636148068921376768,243236955893487616,1.024326e-13,5.116459e-13,0.999967,1.583095e-07,0.000033,0.000000,0.0,0.995962,...,0.813045,0.000000,-2377.143555,0.185107,0.017585,1,<NA>,NaN,<NA>,
9,1636148068921376768,243239395434866688,1.022781e-13,5.108740e-13,0.999975,4.727441e-10,0.000025,0.000000,0.0,0.997474,...,1.605500,0.469493,-1703.919922,0.129690,0.035880,1,<NA>,NaN,<NA>,


In [36]:
arr_spl = np.array_split(oba_stars, 20)
df_fi = pd.DataFrame()
clean_jobids()
Gaia.delete_user_table(table_name='tableid')
for i,vv in enumerate(arr_spl):
    print(i,len(vv))
    dfb = get_meta(vv,Gaia,cols_db)
    df_fi = pd.concat((df_fi,dfb))
df_fi


/home/philippe/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


job ids []
INFO: Removed jobs: '[]'. [astroquery.utils.tap.core]
INFO: Table 'tableid' deleted. [astroquery.utils.tap.core]
0 137347
INFO: Sending pytable. [astroquery.utils.tap.core]
INFO: Uploaded table 'tableid'. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]
INFO: Table 'tableid' deleted. [astroquery.utils.tap.core]
1 137347
INFO: Sending pytable. [astroquery.utils.tap.core]
INFO: Uploaded table 'tableid'. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]
INFO: Table 'tableid' deleted. [astroquery.utils.tap.core]
2 137347
INFO: Sending pytable. [astroquery.utils.tap.core]
INFO: Uploaded table 'tableid'. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]
INFO: Table 'tableid' deleted. [astroquery.utils.tap.core]
3 137347
INFO: Sending pytable. [astroquery.utils.tap.core]
INFO: Uploaded table 'tableid'. [astroquery.utils.tap.core]


...

INFO: Query finished. [astroquery.utils.tap.core]
INFO: Table 'tableid' deleted. [astroquery.utils.tap.core]
17 137346
INFO: Sending pytable. [astroquery.utils.tap.core]
INFO: Uploaded table 'tableid'. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]
INFO: Table 'tableid' deleted. [astroquery.utils.tap.core]
18 137346
INFO: Sending pytable. [astroquery.utils.tap.core]
INFO: Uploaded table 'tableid'. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]
INFO: Table 'tableid' deleted. [astroquery.utils.tap.core]
19 137346
INFO: Sending pytable. [astroquery.utils.tap.core]
INFO: Uploaded table 'tableid'. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]
INFO: Table 'tableid' deleted. [astroquery.utils.tap.core]


,solution_id,source_id,classprob_dsc_combmod_quasar,classprob_dsc_combmod_galaxy,classprob_dsc_combmod_star,classprob_dsc_combmod_whitedwarf,classprob_dsc_combmod_binarystar,classprob_dsc_specmod_quasar,classprob_dsc_specmod_galaxy,classprob_dsc_specmod_star,...,logg_msc2,logg_msc2_upper,logg_msc2_lower,ag_msc,ag_msc_upper,ag_msc_lower,logposterior_msc,mcmcaccept_msc,mcmcdrift_msc,flags_msc
0,1636148068921376768,243212732278284416,1.022396e-13,5.106816e-13,0.999979,5.342136e-10,0.000021,0.000000,0.000000,0.997854,...,4.197229,4.721307,4.105352,0.361903,0.444588,0.122586,-176.645203,0.174511,0.067618,0
1,1636148068921376768,243215068740428672,5.124280e-11,5.103790e-13,0.999985,2.387608e-10,0.000015,0.000005,0.000000,0.998451,...,3.874521,5.200000,2.000000,0.882090,5.000000,0.000000,-698.042114,0.131050,0.010333,0
2,1636148068921376768,243222662242499840,2.685261e-11,5.104969e-13,0.999982,3.322484e-10,0.000018,0.000003,0.000000,0.998218,...,3.949830,5.025512,3.432509,1.242911,1.968054,0.506835,-334.070557,0.115704,0.016557,0
3,1636148068921376768,243225651539759104,1.023207e-13,5.110869e-13,0.999970,8.197416e-10,0.000030,0.000000,0.000000,0.997054,...,4.159162,4.915750,3.717377,0.624483,0.956783,0.493951,-1213.568848,0.168067,0.059471,1
4,1636148068921376768,243225685899488640,1.200140e-10,5.122913e-13,0.999947,9.994340e-09,0.000053,0.000012,0.000000,0.994687,...,3.951599,4.821571,3.581628,0.076618,0.241910,0.000000,-977.054382,0.199928,0.046965,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137341,1636148068921376768,5959547056346917248,1.020557e-13,5.097633e-13,0.999997,2.003229e-12,0.000003,0.000000,0.000000,0.999669,...,4.266311,4.423877,4.109059,0.315463,0.329963,0.291800,-2549.920410,0.266874,0.704024,1
137342,1636148068921376768,5959547567393419392,1.020413e-13,5.096971e-13,0.999998,2.002944e-12,0.000002,0.000000,0.000000,0.999813,...,5.049195,5.200000,4.324937,0.938304,1.118404,0.813499,-662.849121,0.284200,0.229424,0
137343,1636148068921376768,5959547670472310528,1.512998e-10,5.106537e-13,0.999979,1.215285e-09,0.000021,0.000015,0.000000,0.997909,...,3.561956,5.200000,3.261623,0.174846,1.901548,0.000000,-9456.500000,0.155943,0.037604,1
137344,1636148068921376768,5959549255370331392,1.020397e-13,6.026252e-11,0.999998,2.002910e-12,0.000002,0.000000,0.000001,0.999830,...,5.004127,5.200000,3.441970,1.119420,1.566388,0.585613,-51.275024,0.235227,0.107411,0


In [50]:
#df_fi['evolstage_flame'] = df_fi['evolstage_flame'].astype(int)
#df_fi["evolstage_flame_spec"] = df_fi["evolstage_flame_spec"].apply(lambda x:-1 if x is pd.NA else x)
#print(rr_cp['evolstage_flame_spec'].unique())
df_fi = df_fi.fillna(int(-999))
df_fi["spectraltype_esphs"] = df_fi["spectraltype_esphs"].apply(lambda x:'U' if x=='' else x)
#print(rr_cp['spectraltype_esphs'].unique())
df_fi['spectraltype_esphs'] = df_fi['spectraltype_esphs'].astype(str)
for vv in df_fi.dtypes:
    print(type(vv))
#save_df(df_fi,'gaiadr3','golden_sample_oba_astrophysical_parameters')

<class 'numpy.dtypes.Int64DType'>
<class 'numpy.dtypes.Int64DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DType'>
<class 'numpy.dtypes.Float32DTyp

In [14]:
targets

,source_id,ra,dec,pmra,pmdec,parallax,parallax_error,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,l,b,phot_variable_flag,ref_epoch,L,B,ag_gspphot,MG
12,423289063054369024,0.400430,59.672497,-4.377066,-1.054522,0.457445,0.020055,14.929647,15.329857,14.351916,116.671442,-2.592115,NOT_AVAILABLE,2016.0,116.671442,-2.592115,1.3218,1.909540
36,423289509730936576,0.364285,59.702281,-1.777071,-0.541931,0.333426,0.012067,13.022193,13.334257,12.532269,116.659260,-2.559376,NOT_AVAILABLE,2016.0,116.659260,-2.559376,NaN,NaN
44,423289681529592704,0.345458,59.742632,-3.021024,-2.500900,0.348504,0.014043,14.087346,14.365150,13.647771,116.657728,-2.517951,NOT_AVAILABLE,2016.0,116.657728,-2.517951,1.0354,0.762985
46,423289681529603328,0.377291,59.734461,-2.464894,-1.957056,0.286316,0.012006,12.733460,12.903208,12.425391,116.671907,-2.529062,NOT_AVAILABLE,2016.0,116.671907,-2.529062,1.2705,-1.252813
49,423289715889351808,0.324619,59.714754,-2.017454,-1.542614,0.319811,0.021189,15.105708,15.504957,14.505490,116.642019,-2.543275,NOT_AVAILABLE,2016.0,116.642019,-2.543275,1.8370,0.793179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
896811,1997464292947011328,354.039307,55.493297,2.533718,0.973578,0.408756,0.022502,15.054265,15.326829,14.619435,112.346496,-5.823654,NOT_AVAILABLE,2016.0,112.346496,-5.823654,0.7712,2.340383
896870,1997468175597382016,354.161792,55.596995,-1.126363,-0.618143,0.726699,0.016450,13.568130,13.862082,13.105453,112.443188,-5.744274,NOT_AVAILABLE,2016.0,112.443188,-5.744274,0.8300,2.044904
896938,1997481163578395264,354.102056,55.665601,-2.625141,-2.146707,0.456080,0.020978,15.021451,15.351855,14.516724,112.430525,-5.668873,NOT_AVAILABLE,2016.0,112.430525,-5.668873,1.3510,1.965656
896987,1997485802142965248,354.047289,55.765659,2.938829,-2.523933,1.237782,0.013845,11.613557,11.811242,11.278930,112.429792,-5.564170,NOT_AVAILABLE,2016.0,112.429792,-5.564170,0.5411,1.535677


In [15]:
targets.to_hdf('targets_OBA.hdf5',key='gaiadr3')

In [16]:
sel.to_hdf('OBA_gloden.hdf5',key='gaiadr3')